In [ ]:
from src.requirements import *
from src.ssl_large import *
from src.tokenizer import *
import csv, random

In [ ]:
class MemoryMappedTripletDataset(Dataset):
    def __init__(self, triplets, cache_dir='data/cache_mmap_triplet', top_db=20):
        super().__init__()
        self.triplets = triplets
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.top_db = top_db
        
        # Generate unique cache name based on triplet list
        # (so different train/val splits get different caches)
        triplet_hash = hash(str(sorted([t[0] for t in triplets[:100]])))
        cache_name = f"triplet_cache_{len(triplets)}_{abs(triplet_hash)}"
        
        meta_file = self.cache_dir / f"{cache_name}_meta_abx.npz"
        audio_file = self.cache_dir / f"{cache_name}_audio_abx.dat"
        
        if not meta_file.exists():
            print("Preprocessing triplet audio (first time only)...")
            self._preprocess_all(audio_file, meta_file)
        else:
            print(f"Loading metadata from cache...")
        
        # Load metadata
        meta = np.load(meta_file, allow_pickle=True)
        self.audio_shapes = meta['shapes']
        self.audio_offsets = meta['offsets']
        self.total_size = meta['total_size'].item()
        self.path_to_idx = meta['path_to_idx'].item()  # Dictionary mapping paths to indices
        
        # Memory-map audio data
        self.audio_mmap = np.memmap(audio_file, dtype='float32', mode='r', shape=(self.total_size,))
        
        print(f"✓ Cache ready! {len(self.triplets)} triplets")
        print(f"  Total audio size: {self.total_size * 4 / (1024**3):.2f} GB")
        print(f"  Unique audio files: {len(self.path_to_idx)}")
        print(f"  Memory usage: ~0 MB (memory-mapped)")
    
    def _preprocess_single(self, path):
        """Preprocess a single audio file."""
        waveform, sr = sf.read(path, always_2d=True)
        waveform = np.array(waveform.T, dtype=np.float32)
        
        # Stereo to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(axis=0)
        else:
            waveform = waveform[0]
        
        # Trim silence
        trimmed, _ = librosa.effects.trim(waveform, top_db=self.top_db)
        
        # Normalize
        max_val = np.abs(trimmed).max()
        if max_val > 0:
            trimmed = trimmed / max_val
        
        return trimmed.astype(np.float32)
    
    def _preprocess_all(self, audio_file, meta_file):
        """Preprocess all unique audio files and save to memory-mapped file."""
        import time
        start = time.time()
        
        print("Collecting unique audio paths...")
        
        # Get all unique audio paths from triplets
        unique_paths = set()
        for path_a, path_b, path_x, label in self.triplets:
            unique_paths.add(path_a)
            unique_paths.add(path_b)
            unique_paths.add(path_x)
        
        unique_paths = sorted(list(unique_paths))
        print(f"Found {len(unique_paths)} unique audio files")
        
        print("Processing audio files...")
        
        # Process all unique files
        all_audio_data = []
        path_to_idx = {}  # Map path to index in cache
        
        for idx, path in enumerate(tqdm(unique_paths, desc="Processing")):
            try:
                audio = self._preprocess_single(path)
                all_audio_data.append(audio)
                path_to_idx[path] = idx
            except Exception as e:
                print(f"\n❌ Error processing {path}: {e}")
                raise
        
        # Calculate shapes and offsets
        all_shapes = [len(audio) for audio in all_audio_data]
        total_size = sum(all_shapes)
        
        print(f"Total samples: {total_size:,} ({total_size * 4 / (1024**3):.2f} GB)")
        
        # Create offsets
        offsets = np.zeros(len(all_shapes) + 1, dtype=np.int64)
        np.cumsum(all_shapes, out=offsets[1:])
        
        # Write to memory-mapped file
        print("Writing to memory-mapped file...")
        mmap = np.memmap(audio_file, dtype='float32', mode='w+', shape=(total_size,))
        
        for i in tqdm(range(len(all_audio_data)), desc="Writing"):
            start_pos = offsets[i]
            end_pos = offsets[i + 1]
            mmap[start_pos:end_pos] = all_audio_data[i]
        
        mmap.flush()
        del mmap
        del all_audio_data
        
        # Save metadata
        np.savez(meta_file,
                 shapes=np.array(all_shapes, dtype=np.int64),
                 offsets=offsets,
                 total_size=np.array(total_size, dtype=np.int64),
                 path_to_idx=np.array(path_to_idx, dtype=object))
        
        elapsed = time.time() - start
        print(f"✓ Done in {elapsed:.1f}s ({elapsed/60:.1f} min)")
    
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        """Load triplet from memory-mapped cache."""
        path_a, path_b, path_x, label = self.triplets[idx]
        
        # Get audio indices in cache
        idx_a = self.path_to_idx[path_a]
        idx_b = self.path_to_idx[path_b]
        idx_x = self.path_to_idx[path_x]
        
        # Load from memory-mapped file
        start_a = self.audio_offsets[idx_a]
        end_a = self.audio_offsets[idx_a + 1]
        wave_a = torch.from_numpy(self.audio_mmap[start_a:end_a].copy())
        
        start_b = self.audio_offsets[idx_b]
        end_b = self.audio_offsets[idx_b + 1]
        wave_b = torch.from_numpy(self.audio_mmap[start_b:end_b].copy())
        
        start_x = self.audio_offsets[idx_x]
        end_x = self.audio_offsets[idx_x + 1]
        wave_x = torch.from_numpy(self.audio_mmap[start_x:end_x].copy())
        
        return wave_a, wave_b, wave_x, label


def collate_triplet(batch):
    """
    Collate function for triplet dataset.
    
    Args:
        batch: List of (wave_a, wave_b, wave_x, label) tuples
    
    Returns:
        waves_a: (batch, 1, max_len_a)
        waves_b: (batch, 1, max_len_b)
        waves_x: (batch, 1, max_len_x)
        labels: (batch,)
    """
    waves_a, waves_b, waves_x, labels = zip(*batch)
    
    # Ensure all are 1D
    waves_a = [w.flatten() if w.dim() > 1 else w for w in waves_a]
    waves_b = [w.flatten() if w.dim() > 1 else w for w in waves_b]
    waves_x = [w.flatten() if w.dim() > 1 else w for w in waves_x]
    
    # Pad sequences
    waves_a_padded = rnn_utils.pad_sequence(waves_a, batch_first=True, padding_value=0.0)
    waves_b_padded = rnn_utils.pad_sequence(waves_b, batch_first=True, padding_value=0.0)
    waves_x_padded = rnn_utils.pad_sequence(waves_x, batch_first=True, padding_value=0.0)
    
    # Add channel dimension
    waves_a_padded = waves_a_padded.unsqueeze(1)
    waves_b_padded = waves_b_padded.unsqueeze(1)
    waves_x_padded = waves_x_padded.unsqueeze(1)
    
    # Convert labels to tensor
    labels = torch.tensor(labels, dtype=torch.long)
    
    return waves_a_padded, waves_b_padded, waves_x_padded, labels

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, triplets):
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        path_a, path_b, path_x, label = self.triplets[idx]
        
        wave_a = load_wave(path_a)
        wave_b = load_wave(path_b)
        wave_x = load_wave(path_x)

        return wave_a, wave_b, wave_x, label

def collate_fn(batch):
    waves_a, waves_b, waves_x, labels = zip(*batch)
    
    waves_a = rnn_utils.pad_sequence(waves_a, batch_first=True)
    waves_b = rnn_utils.pad_sequence(waves_b, batch_first=True)
    waves_x = rnn_utils.pad_sequence(waves_x, batch_first=True)
    
    return waves_a.unsqueeze(1), waves_b.unsqueeze(1), waves_x.unsqueeze(1), torch.tensor(labels)

In [ ]:
@torch.no_grad()
def abx_test_batched(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for batch_a, batch_b, batch_x, labels in loader:
        batch_a, batch_b, batch_x = batch_a.to(device), batch_b.to(device), batch_x.to(device)
        labels = labels.to(device)

        emb_a = model.extract_features(batch_a).mean(dim=1)
        emb_b = model.extract_features(batch_b).mean(dim=1)
        emb_x = model.extract_features(batch_x).mean(dim=1)

        sim_xa = F.cosine_similarity(emb_x, emb_a)
        sim_xb = F.cosine_similarity(emb_x, emb_b)

        # If label is 0: we want sim_xa > sim_xb
        # If label is 1: we want sim_xb > sim_xa
        is_match_a = (labels == 0)
        
        # Where label is 0, check if X is closer to A. 
        # Where label is 1, check if X is closer to B.
        pred_correct = torch.where(is_match_a, sim_xa >= sim_xb, sim_xb >= sim_xa)
        
        correct += pred_correct.sum().item()
        total += labels.size(0)

    return correct / total

In [ ]:
def load_tsv_dataset(tsv_path):    
    df = pd.read_csv(tsv_path, sep='\t')
    
    print(f"Loaded TSV with columns: {df.columns.tolist()}")
    print(f"Shape: {df.shape}")
    
    if 'path' not in df.columns or 'transcript' not in df.columns:
        raise ValueError(f"Expected columns 'path' and 'transcript', got {df.columns.tolist()}")
    
    dataset = []
    for _, row in df.iterrows():
        dataset.append({
            "audio_path": str(row['path']).strip(),
            "text": str(row['transcript']).strip()
        })
    
    print(f"✓ Loaded {len(dataset)} samples")
    
    if dataset:
        print(f"\nFirst sample:")
        print(f"  Audio: {dataset[0]['audio_path']}")
        print(f"  Text: {dataset[0]['text'][:100]}")
    
    return dataset

In [ ]:
def tokenize_chars(tokenizer, text):
    return tokenizer.encode(text)

In [ ]:
def find_minimal_pairs(dataset, tokenizer, max_pairs=2000):
    tokenized = []
    for i, item in enumerate(dataset):
        tokens = tokenize_chars(tokenizer, item["text"])
        tokenized.append((i, tokens))

    pairs = []
    for i in range(len(tokenized)):
        idx_i, tok_i = tokenized[i]
        for j in range(i + 1, len(tokenized)):
            idx_j, tok_j = tokenized[j]

            if len(tok_i) != len(tok_j):
                continue

            diffs = [k for k in range(len(tok_i)) if tok_i[k] != tok_j[k]]
            if len(diffs) == 1:
                pairs.append((idx_i, idx_j, diffs[0]))

            if len(pairs) >= max_pairs:
                return pairs

    return pairs

In [ ]:
def load_wave(path, target_sr=16000):
    waveform, sr = sf.read(path, always_2d=True)
    waveform = torch.tensor(waveform, dtype=torch.float32)

    if waveform.ndim == 2:
        waveform = waveform.T
        waveform = waveform.mean(dim=0, keepdim=True)
    elif waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)

    wave_np = waveform.squeeze(0).numpy()
    trimmed, _ = librosa.effects.trim(wave_np, top_db=TOP_DB)
    waveform = torch.tensor(trimmed, dtype=torch.float32).unsqueeze(0)
        
    max_val = waveform.abs().max()
    if max_val > 0:
        waveform = waveform / max_val

    if sr != 16_000:
        waveform = torchaudio.functional.resample(waveform, sr, 16_000)

    waveform = waveform.squeeze(0)
    return waveform

In [ ]:
def generate_abx_triplets(dataset, tokenizer, max_triplets=500):
    minimal_pairs = find_minimal_pairs(dataset, tokenizer)
    triplets = []

    text_to_entries = {}
    for d in dataset:
        text_to_entries.setdefault(d["text"], []).append(d)

    for idx_A, idx_B, _ in minimal_pairs:
        A = dataset[idx_A]
        B = dataset[idx_B]

        candidates_A = [d for d in text_to_entries[A["text"]] if d["audio_path"] != A["audio_path"]]
        candidates_B = [d for d in text_to_entries[B["text"]] if d["audio_path"] != B["audio_path"]]

        if candidates_A:
            X = random.choice(candidates_A)
            label = 0  # X matches A
        elif candidates_B:
            X = random.choice(candidates_B)
            label = 1  # X matches B
        else:
            continue

        # wave_A = load_wave(A["audio_path"])
        # wave_B = load_wave(B["audio_path"])
        # wave_X = load_wave(X["audio_path"])

        triplets.append((A["audio_path"], B["audio_path"], X["audio_path"], label))

        # triplets.append((wave_A, wave_B, wave_X, label))

        if len(triplets) >= max_triplets:
            break

    return triplets

In [ ]:
tokenizer = Tokenizer.load(os.path.join("data", "tokenizer.json"))
dataset = load_tsv_dataset(os.path.join("data", "metadata_normal.tsv"))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
update_ver = 25_000
checkpoint_dict = torch.load(os.path.join('models', 'ssl_model', f'ssl_model_checkpoint_{update_ver}.pth'))
ssl_state_dict = checkpoint_dict['model_state_dict']
ssl_model = LargeSSLModel().to(device)
ssl_model.load_state_dict(ssl_state_dict, strict=True)

In [ ]:
cache_dir = os.path.join("data", "cache_mmap", "abx")
triplets = generate_abx_triplets(dataset, tokenizer, max_triplets=500)

abx_triplet_ds = MemoryMappedTripletDataset(
    triplets=triplets,
    cache_dir=cache_dir,
    top_db=TOP_DB
)

abx_loader = DataLoader(
    abx_triplet_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_triplet,
    pin_memory=True,
)

# abx_triplet_ds = TripletDataset(triplets)
# abx_loader = DataLoader(abx_triplet_ds, batch_size=16, collate_fn=collate_fn)

accuracy = abx_test_batched(ssl_model, abx_loader, device)

print(f"Final ABX Score: {accuracy * 100:.2f}%")